In [1]:
import sys
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import struct


import funciones_aux as fau

import funciones_dsa as fun_dsa
import funciones_dsa_unilateral as fun_dsa_u
import funciones_dsa_bilateral as fun_dsa_b

import funciones_plot_dsa as fun_plot

from scipy.signal import welch
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from scipy.stats import pearsonr, spearmanr


# 1. Carga de datos BIS

## 1. 1.  Archivo f_a

In [2]:
ruta_base_advanced = "../data/data_bis_advanced"

archivos_fa = fau.localizar_archivos(ruta_base_advanced, "DH*", "*.f_a")
ruta_fa = archivos_fa[0]
print("Archivo seleccionado:", ruta_fa)

tiempo_fa_unilat, dsa_unilat = fau.cargar_fa_directo(ruta_fa, escalar_db=True)

print("Dimensiones de la matriz:", dsa_unilat.shape)
print("Frecuencias:", dsa_unilat.columns.min(), "a", dsa_unilat.columns.max(), "Hz")

Se han encontrado 3 archivos *.f_a
Archivo seleccionado: ../data/data_bis_advanced\bilateral\M-Py5D-04301923\DH04301923\L04301923.f_a
Dimensiones de la matriz: (60890, 60)
Frecuencias: 0.5 a 30.0 Hz


## 1. 2. Archivo spa

In [6]:
ruta_spa_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.spa"
ruta_ha_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.h_a"



df_spa_raw = fau.procesar_spa(ruta_spa_unilat)
df_spa_unilat = fun_dsa_u.limpiar_spa_unilateral(df_spa_raw)

### Unión de los 2

In [7]:
df_merge_hor = fun_dsa.alinear_spa_con_tiempo(tiempo_fa_unilat, df_spa_unilat)
sef_hor = df_merge_hor["SEF08"]
mf_hor = df_merge_hor["MEDFRQ08"]

dsa_plot_hor, mask_total_hor = fun_dsa.preparar_dsa_con_mask(tiempo_fa_unilat, 
                                                             dsa_unilat, 
                                                             df_merge_hor)


## 1. 3. Archivo .r2a

In [8]:
num_canales, fs, pendiente, offset = fau.extraer_parametros_eeg(ruta_ha_unilat)
print("Parámetros extraídos con éxito:")
print(f" - Canales: {num_canales}")
print(f" - Frecuencia (Hz): {fs}")
print(f" - Pendiente (m): {pendiente:.8f}")
print(f" - Offset (b): {offset:.4f}")

Parámetros extraídos con éxito:
 - Canales: 2
 - Frecuencia (Hz): 128
 - Pendiente (m): 0.05000000
 - Offset (b): -3234.0000


In [9]:
archivo_r2a = r"../data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.r2a"

In [10]:
df_eeg = fun_dsa_u.leer_r2a(
    archivo_r2a,
    pendiente,
    offset,
    fs=fs
)

# 2. Reconstrucción desde EEG crudo

In [17]:
# se pierde un segundo por el uso de ventanas de 2 segundos con etiqueta temporal en el centro
""" 
La función no calcula una DSA para cada muestra aislada ni para cada segundo independiente. 
Calcula una DSA por ventanas completas de 2 segundos (usando 256 muestras).
La ventana avanza cada segundo (cada 128 muestras).

Resultado: Cada fila de la DSA final no sale de un único segundo, sino de una ventana de 2 segundos.


Por eso, con ventanas de 2 segundos y paso de 1 segundo, si tienes un registro de N segundos, se obtienen aproximadamente:
N - 1 filas espectrales. 
No se obtienen N filas, porque no se puede calcular una ventana completa de 2 segundos centrada 
en todos los segundos extremos sin salirse del registro.
"""

df_dsa_canal1, frecuencias_c1 = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg,
    "canal_1_uV",
    fs=128,
    ventana_seg=2,
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="centro"
)

df_dsa_canal2, _ = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg,
    "canal_2_uV",
    fs=128,
    ventana_seg=2,
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad", 
    tiempo_referencia="centro"
)

# -------------------- Depende del modo que pongamos las uds son uV^2 o uV^2/Hz (uds. potencia)--------------------

# columnas del dF generado después de la conversión por FFT -> las frecuencias
cols_freq = [c for c in df_dsa_canal1.columns if c != "tiempo_s"]

# copia del dF del canal 1 para tener esa estructura
df_dsa_media = df_dsa_canal1.copy()

# media (en float) de las potencias (en uV) de los 2 canales
# en función de las frecuencias 
pot_media = (
    df_dsa_canal1[cols_freq].to_numpy(dtype=float) +
    df_dsa_canal2[cols_freq].to_numpy(dtype=float)
) / 2

# ----------- CONVERSIÓN A DECIBELIOS (ref documentación del BIS) -----------------------------------
ref_potencia = 0.0001
df_dsa_media[cols_freq] = 10 * np.log10(
    (pot_media + 1e-12) / (ref_potencia**2)
)

In [18]:
print(np.nanmin(df_dsa_media[cols_freq].values))
print(np.nanmax(df_dsa_media[cols_freq].values))

print(np.nanpercentile(df_dsa_media[cols_freq].values, 2))
print(np.nanpercentile(df_dsa_media[cols_freq].values, 99.5))

43.70621522483941
134.3701940383069
73.19175587586552
117.14144607928772


In [2]:
ruta_base_advanced = "../data/data_bis_advanced"

archivos_fa = fau.localizar_archivos(ruta_base_advanced, "DH*", "*.f_a")
ruta_fa = archivos_fa[0]
print("Archivo seleccionado:", ruta_fa)

tiempo_fa_unilat, dsa_unilat = fau.cargar_fa_directo(ruta_fa, escalar_db=True)

print("Dimensiones de la matriz:", dsa_unilat.shape)
print("Frecuencias:", dsa_unilat.columns.min(), "a", dsa_unilat.columns.max(), "Hz")

Se han encontrado 3 archivos *.f_a
Archivo seleccionado: ../data/data_bis_advanced\bilateral\M-Py5D-04301923\DH04301923\L04301923.f_a
Dimensiones de la matriz: (60890, 60)
Frecuencias: 0.5 a 30.0 Hz


In [19]:
cols_freq_fa = [c for c in dsa_plot_hor.columns]


print(np.nanmin(dsa_plot_hor[cols_freq_fa].values))
print(np.nanmax(dsa_plot_hor[cols_freq_fa].values))

print(np.nanpercentile(dsa_plot_hor[cols_freq_fa].values, 2))
print(np.nanpercentile(dsa_plot_hor[cols_freq_fa].values, 99.5))

C:\Users\usuario\AppData\Local\Temp\ipykernel_15040\2625646359.py:4: RuntimeWarning: All-NaN slice encountered
  print(np.nanmin(dsa_plot_hor[cols_freq_fa].values))
C:\Users\usuario\AppData\Local\Temp\ipykernel_15040\2625646359.py:5: RuntimeWarning: All-NaN slice encountered
  print(np.nanmax(dsa_plot_hor[cols_freq_fa].values))
C:\Users\usuario\anaconda3\envs\TFG\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1409: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
C:\Users\usuario\anaconda3\envs\TFG\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1409: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


nan
nan
nan
nan


##  2. 1.  Preparación de la DSA original

### 2. 1. 1. Adaptación temporal, máscara y plot de DSA EEG

In [14]:
df_spa_unilat, hora_inicio = fun_dsa.obtener_hora_inicio_desde_spa(
    df_spa_unilat
)

tiempo_eeg, dsa_eeg = fun_dsa.adaptar_dsa_reconstruida_para_plot(
    df_dsa=df_dsa_media,
    frecuencias=frecuencias_c1,
    hora_inicio=hora_inicio,
    insertar_fila_inicial_nan=False
)

df_merge_plot = fun_dsa.alinear_spa_con_tiempo(
    tiempo=tiempo_eeg,
    df_spa=df_spa_unilat
)

_, mask_total = fun_dsa.preparar_dsa_con_mask(
    tiempo=tiempo_eeg,
    dsa=dsa_eeg,
    df_merge=df_merge_plot,
    umbral_sqi=15,
    umbral_ceros=0.9
)

mask_comun = mask_total.copy()

# en las reconstrucciones se utilizan como valores mínimos y máximos 
# los percentiles más ajustados para replicar el color


In [15]:
# DSA original f_a con máscara común

""" 
Copia de la dsa proveniente del f_a para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_fa_plot = dsa_unilat.copy()
dsa_fa_plot.loc[mask_comun.values, :] = np.nan



# DSA reconstruida directa con máscara común

""" 
Copia de la dsa proveniente del eeg para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_eeg_directa_plot = dsa_eeg.copy()
dsa_eeg_directa_plot.loc[mask_comun.values, :] = np.nan

IndexError: Boolean index has wrong length: 2119 instead of 60890

## 2. 2. Preparar escala de color de f_a y eeg reconstruida

In [16]:
# DSA original f_a 

# las matrices que vienen de la f_a suelen mostrar valores entre el 49 y 94
matriz_fa, vmin_fa, vmax_fa, norm_fa, cmap_fa = fun_dsa.preparar_escala_color_dsa(
    dsa_fa_plot,
    vmin=49,
    vmax=94,
    gamma=1
)

# DSA reconstruida EEG

matriz_eeg, vmin_eeg, vmax_eeg, norm_eeg, cmap_eeg = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_directa_plot,
    gamma=0.4
)

NameError: name 'dsa_eeg_directa_plot' is not defined

# 3. Comparación de DSA

## 3. 1. Comprobaciones
### 3. 1. 1. Comprobar que ambas matrices están alineadas

In [ ]:
print("DSA f_a:", dsa_fa_plot.shape)
print("DSA EEG:", dsa_eeg_directa_plot.shape)

print("¿Tiempos iguales?")
print((tiempo_fa_unilat.reset_index(drop=True) == tiempo_eeg.reset_index(drop=True)).all())

print("Rango DSA EEG reconstruida:")
print(np.nanmin(dsa_eeg_directa_plot.values), np.nanmax(dsa_eeg_directa_plot.values))

print("Rango DSA f_a:")
print(np.nanmin(dsa_fa_plot.values), np.nanmax(dsa_fa_plot.values))

### 3. 1. 2. Comparación base

In [ ]:
dsa_eeg_comparacion, dsa_fa_comparacion = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_directa_plot,
    dsa_fa_plot
)

dsa_eeg_z = fun_dsa.zscore_global(dsa_eeg_comparacion)
dsa_fa_z = fun_dsa.zscore_global(dsa_fa_comparacion)

metricas_base = fun_dsa.comparar_dsa_global(
    dsa_eeg_z,
    dsa_fa_z
)

metricas_base

# 4. Optimizaciones

Suavizado temporal de la DSA reconstruida desde EEG

La DSA reconstruida desde el EEG crudo viene de un cálculo directo y produce una DSA con muchos cambios rápidos. 

La DSA del f_a del BIS se ve más suave y no cambia tan bruscamente de segundo a segundo. Sugiere que el BIS podría aplicar algún tipo de promedio temporal interno antes de guardar el f_a.

## 4. 1. Suavizado + shift

In [ ]:
ventanas_sp_smooth = [5, 10, 30, 60]

df_suav_shift = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comparacion,
    dsa_fa_comparacion,
    ventanas_suavizado=ventanas_sp_smooth,
    shifts=range(0, 31)
)

df_suav_shift.sort_values("Pearson", ascending=False).head(7)

In [ ]:
#df_suav_shift.sort_values("Spearman", ascending=False).head(7)

## 4. 1. 2. Tabla resumen final

La variante que mejor aproximó la DSA reconstruida desde EEG crudo a la DSA del archivo `f_a` fue la media de potencias de los dos canales, calculada a partir de densidad espectral de potencia y convertida posteriormente a dB. 

Esto sugiere que el archivo `f_a` incorpora información espectral de ambos canales, así como procesamiento temporal suavizado y un posible retardo asociado al cálculo interno del monitor BIS.

In [ ]:
mejor_pearson = df_suav_shift.sort_values("Pearson", ascending=False).iloc[0]
mejor_spearman = df_suav_shift.sort_values("Spearman", ascending=False).iloc[0]

df_resumen_final = pd.DataFrame([
    {
        "criterio": "Mejor Pearson",
        "suavizado_s": mejor_pearson["suavizado_s"],
        "shift_s": mejor_pearson["shift_s"],
        "Pearson": mejor_pearson["Pearson"],
        "Spearman": mejor_pearson["Spearman"],
        "MAE": mejor_pearson["MAE"],
        "RMSE": mejor_pearson["RMSE"],
    },
    {
        "criterio": "Mejor Spearman",
        "suavizado_s": mejor_spearman["suavizado_s"],
        "shift_s": mejor_spearman["shift_s"],
        "Pearson": mejor_spearman["Pearson"],
        "Spearman": mejor_spearman["Spearman"],
        "MAE": mejor_spearman["MAE"],
        "RMSE": mejor_spearman["RMSE"],
    }
])

df_resumen_final

## 4. 2. Suavizado limpio

In [ ]:
suavizado_final = 30

dsa_eeg_suav = dsa_eeg.copy().rolling(
    window=suavizado_final,
    min_periods=1,
    center=False
).mean()

In [ ]:
# máscara común al final
dsa_eeg_suav_plot = dsa_eeg_suav.copy()
dsa_eeg_suav_plot.loc[mask_comun.values, :] = np.nan

matriz_opt, vmin_opt, vmax_opt, norm_opt, cmap_opt = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_plot,
    gamma=0.25
)

### 4. 2. 1. Comprobación de bandas blancas

In [ ]:
mask_blanca_directa = dsa_eeg_directa_plot.isna().all(axis=1)
mask_blanca_suav = dsa_eeg_suav_plot.isna().all(axis=1)

print("bandas blancas directa vs suavizada")
print((mask_blanca_directa == mask_blanca_suav).all())

print("Diferencias:")
print((mask_blanca_directa != mask_blanca_suav).sum())

## 4. 3.  Al haber recortado 10 segundos


In [ ]:
shift_final = 11

# matriz vacía del mismo tamaño que la DSA suavizada original
dsa_eeg_suav_shift_full = pd.DataFrame(
    np.nan,
    index=dsa_eeg_suav.index,
    columns=dsa_eeg_suav.columns
)

# Desplazar la DSA suavizada hacia delante:
# las filas originales 0:-shift pasan a ocupar las filas shift:
dsa_eeg_suav_shift_full.iloc[shift_final:, :] = dsa_eeg_suav.iloc[:-shift_final, :].to_numpy()

# Aplicar la máscara común sobre la línea temporal original completa
dsa_eeg_suav_shift_full.loc[mask_comun.values, :] = np.nan

# Preparar escala de color
matriz_opt_desfase_full, vmin_opt_desfase_full, vmax_opt_desfase_full, norm_opt_desfase_full, cmap_opt_desfase_full = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_shift_full,
    gamma=0.25
)

print("Tiempo:", len(tiempo_fa_unilat))
print("f_a:", matriz_fa.shape)
print("shift full:", matriz_opt_desfase_full.shape)

## 4. 4. Comprobación forma DSA

In [ ]:
print("DSA f_a:", dsa_fa_plot.shape)
print("DSA EEG:", dsa_eeg_directa_plot.shape)
print("DSA suavizada:", dsa_eeg_suav_plot.shape)
print("DSA shift:", dsa_eeg_suav_shift_full.shape)

print("¿Tiempos iguales?")
print((tiempo_fa_unilat.reset_index(drop=True) == tiempo_eeg.reset_index(drop=True)).all())


print("Rango DSA EEG reconstruida:")
print(np.nanmin(dsa_eeg_directa_plot.values), np.nanmax(dsa_eeg_directa_plot.values))

print("Rango DSA f_a:")
print(np.nanmin(dsa_fa_plot.values), np.nanmax(dsa_fa_plot.values))

### 4. 4. 1. Aclaraciones

```python
df_merge_plot        =  SPA alineado con el eeg
df_merge_hor         =  SPA alineado con f_a

```

```python
df_merge_eeg         =  variante de df_merge_plot con SEF/MEF propios directos

```

```python
df_merge_suav        =  variante de df_merge_plot con SEF/MEF propios suavizados

```

```python
df_merge_suav_shift  =  variante de df_merge_suav con shift
```

## 4. 5. COMPARACIÓN FINAL

In [ ]:
paneles = [
    
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_eeg,
        "norm": norm_eeg,
        "cmap": cmap_eeg,
        "df_merge": df_merge_plot,
        "titulo": "DSA reconstruida desde EEG crudo",
        "etiqueta_colorbar": "Intensidad espectral (dB)",
        "mask_total": mask_comun
    },
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_opt,
        "norm": norm_opt,
        "cmap": cmap_opt,
        "df_merge": df_merge_plot,
        "titulo": "DSA reconstruida suavizada",
        "etiqueta_colorbar": "Intensidad espectral (dB)",
        "mask_total": mask_comun
    },
    {
        "tiempo": tiempo_fa_unilat,
        "frecuencias": dsa_fa_plot.columns.astype(float),
        "matriz": matriz_fa,
        "norm": norm_fa,
        "cmap": cmap_fa,
        "df_merge": df_merge_hor,
        "titulo": "DSA original desde archivo f_a",
        "etiqueta_colorbar": "Intensidad espectral (dB)",
        "mask_total": mask_comun
    },
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_opt_desfase_full,
        "norm": norm_opt_desfase_full,
        "cmap": cmap_opt_desfase_full,
        "df_merge": df_merge_plot,
        "titulo": "DSA suavizada + shift exploratorio",
        "etiqueta_colorbar": "Intensidad espectral (dB)",
        "mask_total": mask_comun
    }
]

fig_grid, axes_grid = fun_dsa.plot_cuadricula_4_dsa(
    paneles,
    titulo_general="Comparación visual de las cuatro matrices DSA"
)

# 5. Cálculo de SEF / MEF propios

## 5. 1. SEF/MEF de referencia del BIS extraídos del .spa

In [ ]:
# Referencia exportada por el monitor BIS en el archivo .spa
sef_fa_spa = df_merge_hor["SEF08"].copy()
mef_fa_spa = df_merge_hor["MEDFRQ08"].copy()

## 5. 2. SEF/MEF de la DSA reconstruida directa desde EEG

In [ ]:
#------------------------ Preparar la potencia lineal media ------------------------------


frecuencias_float = np.array([float(c) for c in cols_freq])

# Matriz de potencia/densidad espectral lineal media de los dos canales.
# Filas = ventanas temporales
# Columnas = frecuencias
# Valores = magnitud espectral lineal, no dB
df_pot_media = pd.DataFrame(
    pot_media,
    columns=frecuencias_float
)

# Asegurar que las columnas estén ordenadas por frecuencia
df_pot_media = df_pot_media.reindex(
    sorted(df_pot_media.columns),
    axis=1
)

frecuencias_float = np.array(df_pot_media.columns, dtype=float)

In [ ]:
def ajustar_longitud_curva(curva, longitud_objetivo, rellenar_inicio=True):
    """
    Ajusta una curva SEF/MEF a la longitud del DataFrame temporal usado para pintar.

    Si faltan filas, rellena con NaN al inicio o al final.
    Si sobran filas, recorta.
    """

    curva = np.asarray(curva, dtype=float)
    longitud_actual = len(curva)
    diferencia = longitud_objetivo - longitud_actual

    if diferencia > 0:
        relleno = np.full(diferencia, np.nan)

        if rellenar_inicio:
            curva = np.r_[relleno, curva]
        else:
            curva = np.r_[curva, relleno]

    elif diferencia < 0:
        curva = curva[:longitud_objetivo]

    return curva

In [ ]:
# (basado en la potencia en uV^2/Hz)
# 1. Calculamos SEF y MEF (que tendrán 2119 o 2118 filas dependiendo de la ventana)
sef_eeg, mef_eeg = fun_dsa.calcular_sef_mef_desde_potencia(
    potencia=df_pot_media.to_numpy(dtype=float),
    frecuencias=frecuencias_float,
    percentil_sef=0.95,
    percentil_mef=0.50
)


# Ajustar longitud a la serie temporal de la DSA reconstruida
longitud_objetivo = len(df_merge_plot)

sef_eeg = ajustar_longitud_curva(
    sef_eeg,
    longitud_objetivo=longitud_objetivo,
    rellenar_inicio=True
)

mef_eeg = ajustar_longitud_curva(
    mef_eeg,
    longitud_objetivo=longitud_objetivo,
    rellenar_inicio=True
)

df_merge_eeg = df_merge_plot.copy()
df_merge_eeg["SEF08"] = sef_eeg
df_merge_eeg["MEDFRQ08"] = mef_eeg

## 5. 3. SEF/MEF de la DSA reconstruida suavizada

In [ ]:
suavizado_final = 30

# Suavizado temporal normal sobre la matriz completa.
# La máscara se aplicará al final en la visualización.
df_pot_media_suav = df_pot_media.rolling(
    window=suavizado_final,
    min_periods=1,
    center=False
).mean()

sef_eeg_suav, mef_eeg_suav = fun_dsa.calcular_sef_mef_desde_potencia(
    potencia=df_pot_media_suav.to_numpy(dtype=float),
    frecuencias=frecuencias_float,
    percentil_sef=0.95,
    percentil_mef=0.50
)

sef_eeg_suav = ajustar_longitud_curva(
    sef_eeg_suav,
    longitud_objetivo=longitud_objetivo,
    rellenar_inicio=True
)

mef_eeg_suav = ajustar_longitud_curva(
    mef_eeg_suav,
    longitud_objetivo=longitud_objetivo,
    rellenar_inicio=True
)

df_merge_suav = df_merge_plot.copy()
df_merge_suav["SEF08"] = sef_eeg_suav
df_merge_suav["MEDFRQ08"] = mef_eeg_suav

## 5. 4. SEF/MEF de la DSA suavizada + shift, manteniendo duración original


In [ ]:
"""
calcular SEF/MEF sobre la matriz suavizada y luego desplazar las curvas
"""

shift_final=11

df_merge_suav_shift_full = df_merge_suav.copy()

df_merge_suav_shift_full["SEF08"] = np.nan
df_merge_suav_shift_full["MEDFRQ08"] = np.nan

df_merge_suav_shift_full.loc[shift_final:, "SEF08"] = (df_merge_suav["SEF08"].iloc[:-shift_final].to_numpy())

df_merge_suav_shift_full.loc[shift_final:, "MEDFRQ08"] = (df_merge_suav["MEDFRQ08"].iloc[:-shift_final].to_numpy())

In [ ]:
paneles = [
    
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_eeg,
        "norm": norm_eeg,
        "cmap": cmap_eeg,
        "df_merge": df_merge_eeg,
        "titulo": "DSA reconstruida desde EEG crudo",
        "mask_total": mask_comun,
        "mostrar_sef": True,
        "mostrar_mef": True
    },
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_opt,
        "norm": norm_opt,
        "cmap": cmap_opt,
        "df_merge": df_merge_suav,
        "titulo": "DSA reconstruida suavizada",
        "mask_total": mask_comun,
        "mostrar_sef": True,
        "mostrar_mef": True
    },
    {
        "tiempo": tiempo_fa_unilat,
        "frecuencias": dsa_fa_plot.columns.astype(float),
        "matriz": matriz_fa,
        "norm": norm_fa,
        "cmap": cmap_fa,
        "df_merge": df_merge_hor,
        "titulo": "DSA original desde archivo f_a",
        "mask_total": mask_comun,
        "mostrar_sef": True,
        "mostrar_mef": True
    },
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_opt_desfase_full,
        "norm": norm_opt_desfase_full,
        "cmap": cmap_opt_desfase_full,
        "df_merge": df_merge_suav_shift_full,
        "titulo": "DSA suavizada + shift temporal",
        "mask_total": mask_comun,
        "mostrar_sef": True,
        "mostrar_mef": True
    }
]


fig_grid, axes_grid = fun_dsa.plot_cuadricula_4_dsa(
    paneles,
    titulo_general="Comparación visual de las cuatro matrices DSA"
)

# 6. Potencia por bandas EEG

In [ ]:
# 6. Potencia por bandas EEG

bandas_eeg = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alfa": (8, 13),
    "Beta": (13, 30)
}

ref_uv_rms = 0.0001

## 6. 1. Preparar matrices lineales para bandas

In [ ]:
# ------------------------------------------------------------
# 1. DSA original .f_a
# ------------------------------------------------------------
# dsa_fa_plot está en dB, así que se convierte a potencia lineal.
# Si quieres usar la original sin máscara, cambia dsa_fa_plot por dsa_unilat.

df_pot_fa = (ref_uv_rms ** 2) * (
    10 ** (dsa_fa_plot.astype(float) / 10)
)

df_pot_fa.columns = [float(c) for c in df_pot_fa.columns]

df_pot_fa = df_pot_fa.reindex(
    sorted(df_pot_fa.columns),
    axis=1
)


# ------------------------------------------------------------
# 2. DSA reconstruida suavizada + shift
# ------------------------------------------------------------
# df_pot_media_suav ya viene del punto 5 y está en escala lineal.

df_pot_recon = pd.DataFrame(
    np.nan,
    index=df_pot_media_suav.index,
    columns=df_pot_media_suav.columns
)

df_pot_recon.iloc[shift_final:, :] = (
    df_pot_media_suav.iloc[:-shift_final, :].to_numpy(dtype=float)
)

df_pot_recon.columns = [float(c) for c in df_pot_recon.columns]

df_pot_recon = df_pot_recon.reindex(
    sorted(df_pot_recon.columns),
    axis=1
)

## 6.2. Funciones para bandas desde potencia lineal

In [ ]:
def calcular_bandas_desde_lineal(df_potencia, bandas, ref_uv_rms=0.0001):
    """
    Calcula potencia por bandas EEG desde una matriz lineal.

    Devuelve:
    - df_bandas_lineal: potencia media por Hz en cada banda.
    - df_bandas_db: lo mismo en dB, solo para visualización.
    - df_banda_dominante: banda dominante por instante.
    """

    df = df_potencia.copy()

    frecuencias = np.array([float(c) for c in df.columns])
    orden = np.argsort(frecuencias)

    frecuencias = frecuencias[orden]
    df = df.iloc[:, orden]

    potencia = df.to_numpy(dtype=float)

    if len(frecuencias) > 1:
        df_freq = np.median(np.diff(frecuencias))
    else:
        df_freq = 1.0

    resultados = {}

    for nombre_banda, (fmin, fmax) in bandas.items():

        if nombre_banda == list(bandas.keys())[-1]:
            idx = (frecuencias >= fmin) & (frecuencias <= fmax)
        else:
            idx = (frecuencias >= fmin) & (frecuencias < fmax)

        ancho_banda = fmax - fmin

        if idx.sum() == 0:
            resultados[nombre_banda] = np.full(len(df), np.nan)
            continue

        pot_banda = potencia[:, idx]

        filas_validas = np.isfinite(pot_banda).any(axis=1)

        # Integrar y normalizar por ancho de banda
        pot_integrada = np.nansum(pot_banda, axis=1) * df_freq
        pot_normalizada = pot_integrada / ancho_banda

        pot_normalizada[~filas_validas] = np.nan

        resultados[nombre_banda] = pot_normalizada

    df_bandas_lineal = pd.DataFrame(resultados, index=df.index)

    # Versión en dB solo para visualización
    df_bandas_db = 10 * np.log10(
        df_bandas_lineal / (ref_uv_rms ** 2)
    )

    df_banda_dominante = pd.DataFrame(index=df.index)

    filas_validas = ~df_bandas_lineal.isna().all(axis=1)

    # Columna de texto: Delta, Theta, Alfa, Beta
    df_banda_dominante["banda_dominante"] = pd.Series(
        index=df.index,
        dtype="object"
    )

    df_banda_dominante.loc[filas_validas, "banda_dominante"] = (
        df_bandas_lineal.loc[filas_validas].idxmax(axis=1)
    )

    suma = df_bandas_lineal.sum(axis=1)
    maximo = df_bandas_lineal.max(axis=1)

    df_banda_dominante["proporcion_dominante"] = np.nan

    df_banda_dominante.loc[filas_validas, "proporcion_dominante"] = (
        maximo.loc[filas_validas] / suma.loc[filas_validas]
    )

    return df_bandas_lineal, df_bandas_db, df_banda_dominante

## 6.3. Calcular bandas para .f_a y reconstruida

In [ ]:
df_bandas_lineal_fa, df_bandas_db_fa, df_banda_dominante_fa = (
    calcular_bandas_desde_lineal(
        df_potencia=df_pot_fa,
        bandas=bandas_eeg,
        ref_uv_rms=ref_uv_rms
    )
)

df_bandas_lineal_recon, df_bandas_db_recon, df_banda_dominante_recon = (
    calcular_bandas_desde_lineal(
        df_potencia=df_pot_recon,
        bandas=bandas_eeg,
        ref_uv_rms=ref_uv_rms
    )
)

## 6.4. Aplicar máscara final a las bandas reconstruidas

In [ ]:
# ------------------------------------------------------------
# Aplicar máscara final a las bandas reconstruidas
# ------------------------------------------------------------

mask_np = pd.Series(mask_comun).reset_index(drop=True).astype(bool).to_numpy()

n = min(len(mask_np), len(df_bandas_lineal_recon))

df_bandas_lineal_recon = df_bandas_lineal_recon.reset_index(drop=True)
df_bandas_db_recon = df_bandas_db_recon.reset_index(drop=True)
df_banda_dominante_recon = df_banda_dominante_recon.reset_index(drop=True)

df_bandas_lineal_recon.loc[mask_np[:n], :] = np.nan
df_bandas_db_recon.loc[mask_np[:n], :] = np.nan
df_banda_dominante_recon.loc[mask_np[:n], :] = np.nan

## 6.5. Resumen de predominancia

In [ ]:
def resumen_predominio(df_banda_dominante):
    resumen = (
        df_banda_dominante["banda_dominante"]
        .value_counts(dropna=True)
        .rename_axis("banda")
        .reset_index(name="n_instantes")
    )

    total = df_banda_dominante["banda_dominante"].notna().sum()

    if total > 0:
        resumen["porcentaje"] = 100 * resumen["n_instantes"] / total
    else:
        resumen["porcentaje"] = np.nan

    return resumen


print("Predominio por bandas - DSA original .f_a")
display(resumen_predominio(df_banda_dominante_fa))

print("Predominio por bandas - DSA reconstruida")
display(resumen_predominio(df_banda_dominante_recon))

## 6.6. Proporciones relativas

In [ ]:
df_bandas_relativas_fa = df_bandas_lineal_fa.div(
    df_bandas_lineal_fa.sum(axis=1),
    axis=0
)

df_bandas_relativas_recon = df_bandas_lineal_recon.div(
    df_bandas_lineal_recon.sum(axis=1),
    axis=0
)

comparacion_proporcion_media = pd.DataFrame({
    "f_a_media": df_bandas_relativas_fa.mean(),
    "reconstruida_media": df_bandas_relativas_recon.mean(),
    "diferencia_recon_menos_fa": (
        df_bandas_relativas_recon.mean() -
        df_bandas_relativas_fa.mean()
    )
}).round(4)

display(comparacion_proporcion_media)

## 6.7. Visualización 

In [ ]:
fig, axes = plt.subplots(
    4,
    1,
    figsize=(18, 15),
    sharex=True,
    constrained_layout=True
)

# Potencia por banda en .f_a
for banda in bandas_eeg.keys():
    axes[0].plot(
        df_bandas_db_fa.index,
        df_bandas_db_fa[banda],
        label=banda,
        linewidth=1.3
    )

axes[0].set_title("Potencia temporal por bandas - DSA original .f_a")
axes[0].set_ylabel("Potencia media por Hz\n(dB)")
axes[0].legend(loc="upper right")
axes[0].grid(True, alpha=0.3)


# Potencia por banda en reconstruida
for banda in bandas_eeg.keys():
    axes[1].plot(
        df_bandas_db_recon.index,
        df_bandas_db_recon[banda],
        label=banda,
        linewidth=1.3
    )

axes[1].set_title("Potencia temporal por bandas - DSA reconstruida suavizada + shift")
axes[1].set_ylabel("Potencia media por Hz\n(dB)")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.3)


# Proporción relativa .f_a
axes[2].stackplot(
    df_bandas_relativas_fa.index,
    [df_bandas_relativas_fa[banda] for banda in bandas_eeg.keys()],
    labels=list(bandas_eeg.keys()),
    alpha=0.85
)

axes[2].set_title("Distribución relativa de potencia por bandas - DSA original .f_a")
axes[2].set_ylabel("Proporción relativa")
axes[2].legend(loc="upper right")
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)


# Proporción relativa reconstruida
axes[3].stackplot(
    df_bandas_relativas_recon.index,
    [df_bandas_relativas_recon[banda] for banda in bandas_eeg.keys()],
    labels=list(bandas_eeg.keys()),
    alpha=0.85
)

axes[3].set_title("Distribución relativa de potencia por bandas - DSA reconstruida suavizada + shift")
axes[3].set_ylabel("Proporción relativa")
axes[3].set_xlabel("Tiempo / ventana")
axes[3].legend(loc="upper right")
axes[3].set_ylim(0, 1)
axes[3].grid(True, alpha=0.3)

fig.suptitle(
    "Comparación temporal de bandas EEG: DSA original .f_a vs DSA reconstruida",
    fontsize=16
)

plt.show()

# Pruebas exploratorias

## 1. Reconstrucción optimizada con spectrogram

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import spectrogram

def crear_matriz_dsa_optimizada(
    df_eeg,
    canal,
    fs=128,
    ventana_seg=1,
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="centro"
):
    """
    Crea una matriz tiempo-frecuencia para DSA desde EEG crudo usando spectrogram (100x más rápido).
    """
    # 1. Extraer la señal como array de numpy
    x = df_eeg[canal].to_numpy(dtype=float)

    # 2. Calcular los parámetros de las ventanas en muestras
    nperseg = int(ventana_seg * fs)
    
    # El "noverlap" es cuántas muestras se solapan. 
    # Ej: Ventana de 1s (128), Paso de 1s (128) -> Solapamiento de 0s 
    noverlap = int((ventana_seg - paso_seg) * fs)
    
    # El tamaño de la FFT para lograr la resolución deseada (Zero-Padding)
    nfft = int(fs / paso_freq)

    # 3. Elegir el escalado
    scaling = "density" if modo in ["densidad", "db_densidad"] else "spectrum"

    # =====================================================================
    # 4. EL NÚCLEO OPTIMIZADO (Sustituye a todo el bucle for)
    # spectrogram calcula toda la matriz de golpe en lenguaje C
    # =====================================================================
    f, t_center, Sxx = spectrogram(
        x,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        detrend="constant",
        scaling=scaling
    )

    # Spectrogram devuelve la matriz Sxx como (Frecuencias, Tiempos).
    # Como queremos que las filas sean los tiempos (para el DataFrame), la transponemos (.T)
    Sxx = Sxx.T

    # 5. Filtrar solo las frecuencias deseadas (entre fmin y fmax)
    mascara_freq = (f >= fmin) & (f <= fmax)
    f_sel = f[mascara_freq]
    
    # Seleccionamos todas las filas (tiempos) pero solo las columnas (frecuencias) que nos interesan
    valores_sel = Sxx[:, mascara_freq]

    # 6. Aplicar la conversión matemática a toda la matriz de golpe
    if modo in ["potencia", "densidad"]:
        valores_finales = valores_sel
    elif modo == "amplitud":
        valores_finales = np.sqrt(valores_sel)
    elif modo in ["db", "db_densidad"]:
        # Se suma 1e-12 para evitar el logaritmo de cero
        valores_finales = 10 * np.log10(valores_sel + 1e-12)
    else:
        raise ValueError("modo debe ser 'db', 'db_densidad', 'potencia', 'densidad' o 'amplitud'")

    # 7. Ajustar el vector de tiempos según la referencia elegida
    # spectrogram devuelve 't_center' como el centro exacto de la ventana.
    if tiempo_referencia == "inicio":
        tiempos = t_center - (ventana_seg / 2)
    elif tiempo_referencia == "centro":
        tiempos = t_center
    elif tiempo_referencia == "final":
        tiempos = t_center + (ventana_seg / 2)
    else:
        raise ValueError("tiempo_referencia debe ser 'inicio', 'centro' o 'final'")

    # 8. Construir el DataFrame final
    df_dsa = pd.DataFrame(
        valores_finales,
        columns=[f"{freq:.1f}" for freq in f_sel]
    )
    df_dsa.insert(0, "tiempo_s", tiempos)

    return df_dsa, f_sel

In [ ]:
# --- ADAPTACIÓN DE LAS LLAMADAS ---

df_dsa_canal1_prueba, frecuencias_c1_prueba = crear_matriz_dsa_optimizada(
    df_eeg,
    "canal_1_uV",
    fs=128,
    ventana_seg=1, # Ojo: En tus comentarios hablas de ventana de 2s, ¡asegúrate de poner el número que quieras usar!
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="centro"
)

df_dsa_canal2_prueba, _ = crear_matriz_dsa_optimizada(
    df_eeg,
    "canal_2_uV",
    fs=128,
    ventana_seg=1,
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="centro"
)

# columnas del dF generado después de la conversión por FFT -> las frecuencias
cols_freq_prueba = [c for c in df_dsa_canal1_prueba.columns if c != "tiempo_s"]

# copia del dF del canal 1 para tener esa estructura
df_dsa_media_prueba = df_dsa_canal1_prueba.copy()

# media (en float) de las potencias (en uV) de los 2 canales
# en función de las frecuencias 
pot_media_prueba = (
    df_dsa_canal1_prueba[cols_freq_prueba].to_numpy(dtype=float) +
    df_dsa_canal2_prueba[cols_freq_prueba].to_numpy(dtype=float)
) / 2

# array de los nombres de las columnas pasados a float
frecuencias_float_prueba = np.array([float(c) for c in cols_freq_prueba])

# calcular sef y mef solo para la dsa reconstruida del eeg 
# (basado en la potencia en uV)
# 1. Calculamos SEF y MEF (que tendrán 2119 o 2118 filas dependiendo de la ventana)
sef_eeg_prueba, mef_eeg_prueba = fun_dsa.calcular_sef_mef_desde_potencia(
    potencia=pot_media_prueba,
    frecuencias=frecuencias_float_prueba,
    percentil_sef=0.95,
    percentil_mef=0.50
)

# 2. Averiguamos la longitud objetivo (las filas del DataFrame de tu archivo .spa o .f_a)
# Suponiendo que tienes cargado el dF original del spa/f_a en una variable (ej: df_spa)
longitud_objetivo_prueba = len(df_spa_unilat) 
longitud_actual_prueba = len(sef_eeg_prueba)
filas_faltantes_prueba = longitud_objetivo_prueba - longitud_actual_prueba

# 3. Rellenamos dinámicamente con NaNs
if filas_faltantes_prueba > 0:
    # Creamos un array con tantos NaNs como falten (normalmente 1)
    relleno_nan_prueba = np.full(filas_faltantes_prueba, np.nan)
    
    # Los añadimos al principio
    sef_pad_prueba = np.r_[relleno_nan_prueba, sef_eeg_prueba]
    mef_pad_prueba = np.r_[relleno_nan_prueba, mef_eeg_prueba]
else:
    # Si la ventana es de 1s, filas_faltantes es 0, se queda igual
    sef_pad_prueba = sef_eeg_prueba
    mef_pad_prueba = mef_eeg_prueba

# 4. Creamos las Series finales para plotear
sef_eeg_plot_prueba = pd.Series(sef_pad_prueba, name="SEF08")
mef_eeg_plot_prueba = pd.Series(mef_pad_prueba, name="MEDFRQ08")

# --- CONVERSIÓN A DECIBELIOS (Usando la ref documentada del BIS) ---
ref_potencia = 0.0001
df_dsa_media_prueba[cols_freq_prueba] = 10 * np.log10(
    (pot_media_prueba + 1e-12) / (ref_potencia**2)
)

In [ ]:
# El parámetro 'span' representa los segundos de inercia del filtro.
# El manual del BIS habla de ventanas SpSmooth de 15 o 30 segundos.
df_dsa_suavizada = df_dsa_media_prueba.copy()

# Aplica el filtro exponencial a lo largo del eje temporal (por columnas de frecuencia)
for col in cols_freq_prueba:
    df_dsa_suavizada[col] = df_dsa_media_prueba[col].ewm(span=15, adjust=False).mean()
    
    

df_spa_unilat, hora_inicio = fun_dsa.obtener_hora_inicio_desde_spa(
    df_spa_unilat
)

tiempo_eeg_prueba, dsa_eeg_prueba = fun_dsa.adaptar_dsa_reconstruida_para_plot(
    df_dsa=df_dsa_suavizada,
    frecuencias=frecuencias_c1_prueba,
    hora_inicio=hora_inicio,
    insertar_fila_inicial_nan=False
)

df_merge_plot_prueba = fun_dsa.alinear_spa_con_tiempo(
    tiempo=tiempo_eeg_prueba,
    df_spa=df_spa_unilat
)

df_merge_plot_prueba["SEF08"] = sef_eeg_plot_prueba.values
df_merge_plot_prueba["MEDFRQ08"] = mef_eeg_plot_prueba.values

_, mask_total_prueba = fun_dsa.preparar_dsa_con_mask(
    tiempo=tiempo_eeg_prueba,
    dsa=dsa_eeg_prueba,
    df_merge=df_merge_plot_prueba,
    umbral_sqi=15,
    umbral_ceros=0.9
)

mask_comun_prueba = mask_total_prueba.copy()

# en las reconstrucciones se utilizan como valores mínimos y máximos 
# los percentiles más ajustados para replicar el color


In [ ]:
# DSA reconstruida directa con máscara común

""" 
Copia de la dsa proveniente del eeg para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_eeg_directa_plot_prueba = dsa_eeg_prueba.copy()
dsa_eeg_directa_plot_prueba.loc[mask_comun_prueba.values, :] = np.nan

# Preparar escala de color
matriz_prueba, vmin_prueba, vmax_prueba, norm_prueba, cmap_prueba = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_directa_plot_prueba,
    gamma=0.3
)

In [ ]:
fig_prueba, ax_prueba, ax_band_prueba, cax_prueba = fun_plot.plot_dsa_con_sef_mef(
    tiempo=tiempo_eeg_prueba,
    frecuencias=frecuencias_c1_prueba,
    matriz=matriz_prueba,
    norm=norm_prueba,
    cmap=cmap_prueba,
    df_merge=df_merge_plot_prueba,
    titulo=f"DSA reconstruida:",
    etiqueta_colorbar="Intensidad espectral (dB)",
    mask_total=mask_comun_prueba,
    mostrar_sef=False,
    mostrar_mef=False
)

In [ ]:
cols_freq_prueba = [c for c in dsa_eeg_directa_plot_prueba.columns]


print(np.nanmin(dsa_eeg_directa_plot_prueba[cols_freq_prueba].values))
print(np.nanmax(dsa_eeg_directa_plot_prueba[cols_freq_prueba].values))

print(np.nanpercentile(dsa_eeg_directa_plot_prueba[cols_freq_prueba].values, 2))
print(np.nanpercentile(dsa_eeg_directa_plot_prueba[cols_freq_prueba].values, 99.5))

In [ ]:
# DSA original f_a con máscara común

""" 
Copia de la dsa proveniente del f_a para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_fa_plot = dsa_unilat.copy()
dsa_fa_plot.loc[mask_comun_prueba.values, :] = np.nan

dsa_eeg_comparacion, dsa_fa_comparacion = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_directa_plot_prueba,
    dsa_fa_plot
)

In [ ]:
ventanas_sp_smooth = [1, 5, 10, 30, 60]

df_suav_shift = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comparacion,
    dsa_fa_comparacion,
    ventanas_suavizado=ventanas_sp_smooth,
    shifts=range(0, 61)
)

df_suav_shift.sort_values("Pearson", ascending=False).head(7)

In [ ]:
suavizado_final = 30

dsa_eeg_suav_prueba = dsa_eeg_prueba.copy().rolling(
    window=suavizado_final,
    min_periods=1,
    center=False
).mean()

# máscara común al final
dsa_eeg_suav_plot_prueba = dsa_eeg_suav_prueba.copy()
dsa_eeg_suav_plot_prueba.loc[mask_comun_prueba.values, :] = np.nan

matriz_opt_prueba, vmin_opt_prueba, vmax_opt_prueba, norm_opt_prueba, cmap_opt_prueba = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_plot_prueba,
    gamma=0.25
)

In [ ]:
fig_opt_prueba, ax_opt_prueba, ax_opt_band_prueba, cax_prueba = fun_plot.plot_dsa_con_sef_mef(
    tiempo=tiempo_eeg_prueba,
    frecuencias=frecuencias_c1_prueba,
    matriz=matriz_opt_prueba,
    norm=norm_opt_prueba,
    cmap=cmap_opt_prueba,
    df_merge=df_merge_plot_prueba,
    titulo=f"DSA reconstruida:",
    etiqueta_colorbar="Intensidad espectral (dB)",
    mask_total=mask_comun_prueba,
    mostrar_sef=False,
    mostrar_mef=False
)

In [ ]:
shift_final = 5

# matriz vacía del mismo tamaño que la DSA suavizada original
dsa_eeg_suav_shift_full = pd.DataFrame(
    np.nan,
    index=dsa_eeg_suav_prueba.index,
    columns=dsa_eeg_suav_prueba.columns
)

# Desplazar la DSA suavizada hacia delante:
# las filas originales 0:-shift pasan a ocupar las filas shift:
dsa_eeg_suav_shift_full.iloc[shift_final:, :] = dsa_eeg_suav_prueba.iloc[:-shift_final, :].to_numpy()

# Aplicar la máscara común sobre la línea temporal original completa
dsa_eeg_suav_shift_full.loc[mask_comun_prueba.values, :] = np.nan

# Preparar escala de color
matriz_opt_desfase_full, vmin_opt_desfase_full, vmax_opt_desfase_full, norm_opt_desfase_full, cmap_opt_desfase_full = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_shift_full,
    gamma=0.25
)

print("Tiempo:", len(tiempo_fa_unilat))
print("f_a:", matriz_fa.shape)
print("shift full:", matriz_opt_desfase_full.shape)

In [ ]:
fig_opt_prueba, ax_opt_prueba, ax_opt_band_prueba, cax_prueba = fun_plot.plot_dsa_con_sef_mef(
    tiempo=tiempo_eeg_prueba,
    frecuencias=frecuencias_c1_prueba,
    matriz=matriz_opt_desfase_full,
    norm=norm_opt_desfase_full,
    cmap=cmap_opt_desfase_full,
    df_merge=df_merge_plot_prueba,
    titulo=f"DSA reconstruida:",
    etiqueta_colorbar="Intensidad espectral (dB)",
    mask_total=mask_comun_prueba,
    mostrar_sef=False,
    mostrar_mef=False
)

## 2. Reconstrucción optimizada con wavelets

In [ ]:
#pip install PyWavelets

In [ ]:
import numpy as np
import pandas as pd
import pywt  # Importamos el estándar de wavelets

def crear_matriz_dsa_wavelet(
    df_eeg,
    canal,
    fs=128,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5
):
    """
    Crea una matriz DSA usando Transformada Wavelet Continua (CWT) con PyWavelets.
    """
    # 1. Extraer la señal cruda
    x = df_eeg[canal].to_numpy(dtype=float)
    
    # 2. Definir frecuencias deseadas
    frecuencias = np.arange(fmin, fmax + paso_freq, paso_freq)
    
    # 3. Calcular las escalas exactas para PyWavelets
    # Nombre de la wavelet de Morlet continua en pywt
    wavelet_name = 'morl' 
    
    # Frecuencia central matemática de la Morlet
    fc = pywt.central_frequency(wavelet_name) 
    
    # Ecuación de conversión automática de Frecuencia a Escala
    escalas = fc / (frecuencias * (1 / fs))
    
    # =====================================================================
    # 4. EL NÚCLEO WAVELET (PyWavelets es mucho más rápido que SciPy)
    # =====================================================================
    coeficientes, _ = pywt.cwt(x, escalas, wavelet_name, sampling_period=1/fs)
    
    # 5. Convertir a Potencia Absoluta
    potencia_cwt = np.abs(coeficientes)**2
    
    # 6. Compresión Temporal (Agrupar 128 muestras en 1 segundo)
    num_segundos = len(x) // fs
    potencia_segundos = np.zeros((len(frecuencias), num_segundos))
    
    for i in range(num_segundos):
        inicio = i * fs
        fin = inicio + fs
        # Promediamos la energía de esas 128 muestras para resumir ese segundo
        potencia_segundos[:, i] = np.mean(potencia_cwt[:, inicio:fin], axis=1)
        
    # 7. Transponer y construir el DataFrame final
    matriz_final = potencia_segundos.T
    tiempos = np.arange(num_segundos) + 0.5 
    
    df_dsa = pd.DataFrame(
        matriz_final,
        columns=[f"{freq:.1f}" for freq in frecuencias]
    )
    df_dsa.insert(0, "tiempo_s", tiempos)
    
    return df_dsa, frecuencias

In [ ]:
# --- ADAPTACIÓN DE LAS LLAMADAS ---

df_dsa_canal1_wt, frecuencias_c1_wt = crear_matriz_dsa_wavelet(
    df_eeg,
    canal="canal_1_uV",
    fs=128,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5
)

df_dsa_canal2_wt, _ = crear_matriz_dsa_wavelet(
    df_eeg,
    canal="canal_2_uV",
    fs=128,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5
)

# columnas del dF generado después de la conversión por FFT -> las frecuencias
cols_freq_wt = [c for c in df_dsa_canal1_wt.columns if c != "tiempo_s"]

# copia del dF del canal 1 para tener esa estructura
df_dsa_media_wt = df_dsa_canal1_wt.copy()

# media (en float) de las potencias (en uV) de los 2 canales
# en función de las frecuencias 
pot_media_wt = (
    df_dsa_canal1_wt[cols_freq_wt].to_numpy(dtype=float) +
    df_dsa_canal2_wt[cols_freq_wt].to_numpy(dtype=float)
) / 2

# array de los nombres de las columnas pasados a float
frecuencias_float_wt = np.array([float(c) for c in cols_freq_wt])

# calcular sef y mef solo para la dsa reconstruida del eeg 
# (basado en la potencia en uV)
# 1. Calculamos SEF y MEF (que tendrán 2119 o 2118 filas dependiendo de la ventana)
sef_eeg_wt, mef_eeg_wt = fun_dsa.calcular_sef_mef_desde_potencia(
    potencia=pot_media_wt,
    frecuencias=frecuencias_float_wt,
    percentil_sef=0.95,
    percentil_mef=0.50
)

# 2. Averiguamos la longitud objetivo (las filas del DataFrame de tu archivo .spa o .f_a)
# Suponiendo que tienes cargado el dF original del spa/f_a en una variable (ej: df_spa)
longitud_objetivo_wt = len(df_spa_unilat) 
longitud_actual_wt = len(sef_eeg_wt)
filas_faltantes_wt = longitud_objetivo_wt - longitud_actual_wt

# 3. Rellenamos dinámicamente con NaNs
if filas_faltantes_wt > 0:
    # Creamos un array con tantos NaNs como falten (normalmente 1)
    relleno_nan_wt = np.full(filas_faltantes_wt, np.nan)
    
    # Los añadimos al principio
    sef_pad_wt = np.r_[relleno_nan_wt, sef_eeg_wt]
    mef_pad_wt = np.r_[relleno_nan_wt, mef_eeg_wt]
else:
    # Si la ventana es de 1s, filas_faltantes es 0, se queda igual
    sef_pad_wt = sef_eeg_wt
    mef_pad_wt = mef_eeg_wt

# 4. Creamos las Series finales para plotear
sef_eeg_plot_wt = pd.Series(sef_pad_wt, name="SEF08")
mef_eeg_plot_wt = pd.Series(mef_pad_wt, name="MEDFRQ08")

# --- CONVERSIÓN A DECIBELIOS (Usando la ref documentada del BIS) ---
ref_potencia = 0.0001
df_dsa_media_wt[cols_freq_wt] = 10 * np.log10(
    (pot_media_wt + 1e-12) / (ref_potencia**2)
)

In [ ]:
# El parámetro 'span' representa los segundos de inercia del filtro.
# El manual del BIS habla de ventanas SpSmooth de 15 o 30 segundos.
df_dsa_suavizada_wt = df_dsa_media_wt.copy()

# Aplica el filtro exponencial a lo largo del eje temporal (por columnas de frecuencia)
for col in cols_freq_wt:
    df_dsa_suavizada_wt[col] = df_dsa_media_wt[col].ewm(span=15, adjust=False).mean()
    
    

df_spa_unilat, hora_inicio = fun_dsa.obtener_hora_inicio_desde_spa(
    df_spa_unilat
)

tiempo_eeg_wt, dsa_eeg_wt = fun_dsa.adaptar_dsa_reconstruida_para_plot(
    df_dsa=df_dsa_suavizada_wt,
    frecuencias=frecuencias_c1_wt,
    hora_inicio=hora_inicio,
    insertar_fila_inicial_nan=False
)

df_merge_plot_wt = fun_dsa.alinear_spa_con_tiempo(
    tiempo=tiempo_eeg_wt,
    df_spa=df_spa_unilat
)

df_merge_plot_wt["SEF08"] = sef_eeg_plot_wt.values
df_merge_plot_wt["MEDFRQ08"] = mef_eeg_plot_wt.values

_, mask_total_wt = fun_dsa.preparar_dsa_con_mask(
    tiempo=tiempo_eeg_wt,
    dsa=dsa_eeg_wt,
    df_merge=df_merge_plot_wt,
    umbral_sqi=15,
    umbral_ceros=0.9
)

mask_comun_wt = mask_total_wt.copy()

# en las reconstrucciones se utilizan como valores mínimos y máximos 
# los percentiles más ajustados para replicar el color


In [ ]:
# DSA reconstruida directa con máscara común

""" 
Copia de la dsa proveniente del eeg para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_eeg_directa_plot_wt = dsa_eeg_wt.copy()
dsa_eeg_directa_plot_wt.loc[mask_comun_wt.values, :] = np.nan
 
# Preparar escala de color
matriz_wt, vmin_wt, vmax_wt, norm_wt, cmap_wt = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_directa_plot_wt,
    gamma=0.3
)

In [ ]:
fig_wt, ax_wt, ax_band_wt, cax_wt = fun_plot.plot_dsa_con_sef_mef(
    tiempo=tiempo_eeg_wt,
    frecuencias=frecuencias_c1_wt,
    matriz=matriz_wt,
    norm=norm_wt,
    cmap=cmap_wt,
    df_merge=df_merge_plot_wt,
    titulo=f"DSA reconstruida:",
    etiqueta_colorbar="Intensidad espectral (dB)",
    mask_total=mask_comun_wt,
    mostrar_sef=False,
    mostrar_mef=False
)

In [ ]:
cols_freq_wt = [c for c in dsa_eeg_directa_plot_wt.columns]


print(np.nanmin(dsa_eeg_directa_plot_wt[cols_freq_wt].values))
print(np.nanmax(dsa_eeg_directa_plot_wt[cols_freq_wt].values))

print(np.nanpercentile(dsa_eeg_directa_plot_wt[cols_freq_wt].values, 2))
print(np.nanpercentile(dsa_eeg_directa_plot_wt[cols_freq_wt].values, 99.5))

In [ ]:
# DSA original f_a con máscara común

""" 
Copia de la dsa proveniente del f_a para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_fa_plot = dsa_unilat.copy()
dsa_fa_plot.loc[mask_comun_wt.values, :] = np.nan

dsa_eeg_comparacion_wt, dsa_fa_comparacion = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_directa_plot_wt,
    dsa_fa_plot
)

In [ ]:
ventanas_sp_smooth = [1, 5, 10, 30, 60]

df_suav_shift_wt = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comparacion_wt,
    dsa_fa_comparacion,
    ventanas_suavizado=ventanas_sp_smooth,
    shifts=range(0, 61)
)

df_suav_shift_wt.sort_values("Pearson", ascending=False).head(7)

In [ ]:
suavizado_final = 30

dsa_eeg_suav_wt = dsa_eeg_wt.copy().rolling(
    window=suavizado_final,
    min_periods=1,
    center=False
).mean()

In [ ]:
shift_final = 5

# matriz vacía del mismo tamaño que la DSA suavizada original
dsa_eeg_suav_shift_full_wt = pd.DataFrame(
    np.nan,
    index=dsa_eeg_suav_wt.index,
    columns=dsa_eeg_suav_wt.columns
)

# Desplazar la DSA suavizada hacia delante:
# las filas originales 0:-shift pasan a ocupar las filas shift:
dsa_eeg_suav_shift_full_wt.iloc[shift_final:, :] = dsa_eeg_suav_wt.iloc[:-shift_final, :].to_numpy()

# Aplicar la máscara común sobre la línea temporal original completa
dsa_eeg_suav_shift_full_wt.loc[mask_comun_wt.values, :] = np.nan

# Preparar escala de color
matriz_opt_desfase_full_wt, vmin_opt_desfase_full_wt, vmax_opt_desfase_full_wt, norm_opt_desfase_full_wt, cmap_opt_desfase_full_wt = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_shift_full_wt,
    gamma=0.25
)

print("Tiempo:", len(tiempo_fa_unilat))
print("f_a:", matriz_fa.shape)
print("shift full:", matriz_opt_desfase_full_wt.shape)

In [ ]:
fig_opt_prueba, ax_opt_prueba, ax_opt_band_prueba, cax_prueba = fun_plot.plot_dsa_con_sef_mef(
    tiempo=tiempo_eeg_wt,
    frecuencias=frecuencias_c1_wt,
    matriz=matriz_opt_desfase_full_wt,
    norm=norm_opt_desfase_full_wt,
    cmap=cmap_opt_desfase_full_wt,
    df_merge=df_merge_plot_wt,
    titulo=f"DSA reconstruida:",
    etiqueta_colorbar="Intensidad espectral (dB)",
    mask_total=mask_comun_wt,
    mostrar_sef=False,
    mostrar_mef=False
)

El suavizado de 30 s está justificado por SpSmooth, no elegido a ojo; el retardo no está fijado en manual, pero es coherente con el efecto de un suavizado temporal, así que se estima empíricamente por rejilla. Welch y spectrogram quedan prácticamente empatados; wavelets aporta como exploración, pero no parece prioritaria para el resultado principal.

## Tabla resumen de metricas de comparacion

Esta tabla se genera a partir de las matrices y mascaras calculadas en las celdas anteriores. La referencia es la DSA del archivo `.f_a` y las metricas se calculan sobre matrices normalizadas con z-score global.

In [ ]:
# ============================================================
# TABLA RESUMEN DE METRICAS DSA/TCA
# ============================================================

def obtener_suavizado_desde_sp_smooth(df_spa_original, ventanas=(1, 5, 10, 30, 60), valor_por_defecto=30):
    """Devuelve la ventana de suavizado asociada al codigo SpSmooth del .spa."""
    if "SpSmooth" not in df_spa_original.columns:
        return valor_por_defecto, np.nan

    codigos = pd.to_numeric(df_spa_original["SpSmooth"], errors="coerce").dropna()
    if codigos.empty:
        return valor_por_defecto, np.nan

    codigo = int(codigos.mode().iloc[0])
    if 0 <= codigo < len(ventanas):
        return ventanas[codigo], codigo

    return valor_por_defecto, codigo


def dsa_fa_con_mascara(mask):
    """Reconstruye la DSA de referencia .f_a con la mascara comun indicada."""
    dsa_ref = dsa_unilat.copy()
    dsa_ref.loc[mask.values, :] = np.nan
    return dsa_ref


def metricas_globales(dsa_recon, dsa_ref):
    """Calcula las metricas globales usando el mismo criterio del notebook."""
    A, B = fun_dsa.preparar_matrices_para_comparacion(dsa_recon, dsa_ref)
    A_z = fun_dsa.zscore_global(A)
    B_z = fun_dsa.zscore_global(B)
    return fun_dsa.comparar_dsa_global(A_z, B_z)


def fila_metricas(tratamiento, metodo_base, procesado, dsa_recon, dsa_ref, suavizado_s=None, shift_s=0, criterio="directo", nota=""):
    met = metricas_globales(dsa_recon, dsa_ref)
    return {
        "tratamiento": tratamiento,
        "metodo_base": metodo_base,
        "procesado": procesado,
        "suavizado_s": suavizado_s,
        "shift_s": shift_s,
        "criterio": criterio,
        "n_valores": int(met["n_valores_comparados"]),
        "Pearson": met["Pearson"],
        "Spearman": met["Spearman"],
        "MAE": met["MAE"],
        "RMSE": met["RMSE"],
        "nota": nota,
    }


def pares_para_rejilla(dsa_recon, dsa_ref, suavizado_s, shift_s):
    """Reproduce el recorte temporal usado en probar_suavizado_y_shifts."""
    dsa_suav = dsa_recon.rolling(
        window=int(suavizado_s),
        min_periods=1,
        center=False
    ).mean()

    shift_s = int(shift_s)
    if shift_s < 0:
        A = dsa_suav.iloc[-shift_s:].reset_index(drop=True)
        B = dsa_ref.iloc[:len(A)].reset_index(drop=True)
    elif shift_s > 0:
        A = dsa_suav.iloc[:-shift_s].reset_index(drop=True)
        B = dsa_ref.iloc[shift_s:].reset_index(drop=True)
    else:
        A = dsa_suav.reset_index(drop=True)
        B = dsa_ref.reset_index(drop=True)

    n = min(len(A), len(B))
    return A.iloc[:n], B.iloc[:n]


def n_valores_rejilla(dsa_recon, dsa_ref, suavizado_s, shift_s):
    A, B = pares_para_rejilla(dsa_recon, dsa_ref, suavizado_s, shift_s)
    return int((np.isfinite(A.to_numpy(dtype=float)) & np.isfinite(B.to_numpy(dtype=float))).sum())


def fila_desde_rejilla(tratamiento, metodo_base, procesado, fila_rejilla, dsa_recon, dsa_ref, criterio, nota=""):
    suavizado_s = int(fila_rejilla["suavizado_s"])
    shift_s = int(fila_rejilla["shift_s"])
    return {
        "tratamiento": tratamiento,
        "metodo_base": metodo_base,
        "procesado": procesado,
        "suavizado_s": suavizado_s,
        "shift_s": shift_s,
        "criterio": criterio,
        "n_valores": n_valores_rejilla(dsa_recon, dsa_ref, suavizado_s, shift_s),
        "Pearson": fila_rejilla["Pearson"],
        "Spearman": fila_rejilla["Spearman"],
        "MAE": fila_rejilla["MAE"],
        "RMSE": fila_rejilla["RMSE"],
        "nota": nota,
    }


# Ventana guiada por el campo SpSmooth del archivo .spa.
suavizado_sp_smooth_s, codigo_sp_smooth = obtener_suavizado_desde_sp_smooth(df_spa_unilat)

# ------------------------- Welch original -------------------------
dsa_fa_welch_plot = dsa_fa_con_mascara(mask_comun)

dsa_eeg_suav_sp = dsa_eeg.copy().rolling(
    window=suavizado_sp_smooth_s,
    min_periods=1,
    center=False
).mean()
dsa_eeg_suav_sp.loc[mask_comun.values, :] = np.nan

dsa_eeg_comp_welch, dsa_fa_comp_welch = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_directa_plot,
    dsa_fa_welch_plot
)

df_rejilla_welch = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comp_welch,
    dsa_fa_comp_welch,
    ventanas_suavizado=[5, 10, 30, 60],
    shifts=range(0, 31)
)
mejor_welch_pearson = df_rejilla_welch.sort_values("Pearson", ascending=False).iloc[0]

# ------------------------- Spectrogram -------------------------
dsa_fa_spectrogram_plot = dsa_fa_con_mascara(mask_comun_prueba)

dsa_eeg_comp_spectrogram, dsa_fa_comp_spectrogram = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_directa_plot_prueba,
    dsa_fa_spectrogram_plot
)

df_rejilla_spectrogram = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comp_spectrogram,
    dsa_fa_comp_spectrogram,
    ventanas_suavizado=[1, 5, 10, 30, 60],
    shifts=range(0, 61)
)
mejor_spectrogram_pearson = df_rejilla_spectrogram.sort_values("Pearson", ascending=False).iloc[0]

# ------------------------- Wavelets -------------------------
dsa_fa_wavelets_plot = dsa_fa_con_mascara(mask_comun_wt)

dsa_eeg_comp_wavelets, dsa_fa_comp_wavelets = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_directa_plot_wt,
    dsa_fa_wavelets_plot
)

df_rejilla_wavelets = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comp_wavelets,
    dsa_fa_comp_wavelets,
    ventanas_suavizado=[1, 5, 10, 30, 60],
    shifts=range(0, 61)
)
mejor_wavelets_pearson = df_rejilla_wavelets.sort_values("Pearson", ascending=False).iloc[0]
mejor_wavelets_spearman = df_rejilla_wavelets.sort_values("Spearman", ascending=False).iloc[0]

# ------------------------- Tabla final -------------------------
filas_metricas = [
    fila_metricas(
        "Reconstruccion directa",
        "Welch por ventanas",
        "Media de potencia C1/C2; sin suavizado adicional ni retardo",
        dsa_eeg_directa_plot,
        dsa_fa_welch_plot,
        suavizado_s=None,
        shift_s=0,
        criterio="base",
        nota="Comparacion base contra .f_a"
    ),
    fila_metricas(
        "Suavizado causal segun SpSmooth",
        "Welch por ventanas",
        "Media movil causal sobre la DSA reconstruida",
        dsa_eeg_suav_sp,
        dsa_fa_welch_plot,
        suavizado_s=suavizado_sp_smooth_s,
        shift_s=0,
        criterio="SpSmooth",
        nota=f"SpSmooth={codigo_sp_smooth} -> {suavizado_sp_smooth_s} s"
    ),
    fila_desde_rejilla(
        "Suavizado + retardo optimo",
        "Welch por ventanas",
        "Media movil causal + desplazamiento temporal empirico",
        mejor_welch_pearson,
        dsa_eeg_comp_welch,
        dsa_fa_comp_welch,
        criterio="mejor Pearson",
        nota="Retardo estimado por busqueda en rejilla"
    ),
    fila_metricas(
        "Spectrogram + filtro exponencial",
        "scipy.signal.spectrogram",
        "EWM span=15; sin suavizado extra ni retardo",
        dsa_eeg_directa_plot_prueba,
        dsa_fa_spectrogram_plot,
        suavizado_s=None,
        shift_s=0,
        criterio="base optimizada",
        nota="Tratamiento previo de la prueba spectrogram"
    ),
    fila_desde_rejilla(
        "Spectrogram + suavizado + retardo optimo",
        "scipy.signal.spectrogram",
        "EWM span=15 + media movil causal + desplazamiento temporal empirico",
        mejor_spectrogram_pearson,
        dsa_eeg_comp_spectrogram,
        dsa_fa_comp_spectrogram,
        criterio="mejor Pearson",
        nota="Retardo estimado por busqueda en rejilla"
    ),
    fila_metricas(
        "Wavelets + filtro exponencial",
        "CWT Morlet (PyWavelets)",
        "EWM span=15; sin suavizado extra ni retardo",
        dsa_eeg_directa_plot_wt,
        dsa_fa_wavelets_plot,
        suavizado_s=None,
        shift_s=0,
        criterio="exploratorio",
        nota="Prueba exploratoria"
    ),
    fila_desde_rejilla(
        "Wavelets + mejor Pearson",
        "CWT Morlet (PyWavelets)",
        "EWM span=15 + media movil causal + desplazamiento temporal empirico",
        mejor_wavelets_pearson,
        dsa_eeg_comp_wavelets,
        dsa_fa_comp_wavelets,
        criterio="mejor Pearson",
        nota="No supera Welch/spectrogram"
    ),
    fila_desde_rejilla(
        "Wavelets + mejor Spearman",
        "CWT Morlet (PyWavelets)",
        "EWM span=15 + media movil causal + desplazamiento temporal empirico",
        mejor_wavelets_spearman,
        dsa_eeg_comp_wavelets,
        dsa_fa_comp_wavelets,
        criterio="mejor Spearman",
        nota="Configuracion comparable, pero secundaria"
    ),
]

df_tabla_metricas_dsa = pd.DataFrame(filas_metricas)

columnas_numericas = ["Pearson", "Spearman", "MAE", "RMSE"]
df_tabla_metricas_dsa[columnas_numericas] = df_tabla_metricas_dsa[columnas_numericas].round(6)

df_tabla_metricas_dsa = df_tabla_metricas_dsa[
    [
        "tratamiento",
        "metodo_base",
        "procesado",
        "suavizado_s",
        "shift_s",
        "criterio",
        "n_valores",
        "Pearson",
        "Spearman",
        "MAE",
        "RMSE",
        "nota",
    ]
]

display(df_tabla_metricas_dsa)
